# Week 46, Day 6: Forge: Week 46 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w046/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
print("1.", 0.01 * 5, "rad^2")
print("2.", 1 - 3 / 5)
print("3. about", (60 + 10) / 2, "(the average, because both stimuli join the normalisation pool)")

### Exercise: Part A: drift with and without facilitation

Complete run(): each step, compute rates R = f(X); the effective presynaptic output is R times u / U when stf is on (R otherwise); update X by dt (-X + eff W^T + 0.1); and, with stf on, update u by the facilitation equation. Return each bump's absolute drift from its start.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
N = 60; th = np.linspace(-np.pi, np.pi, N, endpoint=False); D = th[:, None] - th[None, :]
f = lambda x: np.clip(x, 0, 1)

def run(stf, trials=40, steps=6000, het=0.5, seed=0, dt=0.1, tauF=20.0, U=0.3):
    g = np.random.default_rng(seed)
    W = (-2 + 6 * np.cos(D)) / N + het * g.normal(size=(N, N)) / N
    starts = np.linspace(-np.pi, np.pi, trials, endpoint=False)
    X = np.maximum(0, np.cos(th[None] - starts[:, None]))
    u = U + (1 - U) * np.maximum(0, np.cos(th[None] - starts[:, None]))
    for k in range(steps):
        R = f(X)
        ...
    c = np.angle((f(X) * np.exp(1j * th)).sum(1))
    return np.abs(np.angle(np.exp(1j * (c - starts))))

drift = {s: run(s) for s in [False, True]}
for s, d in drift.items():
    print(f"facilitation {'on ' if s else 'off'}: median drift {np.median(d):.3f} rad, largest {d.max():.2f} rad")
plt.hist(drift[False], 20, alpha=0.6, label="no facilitation"); plt.hist(drift[True], 20, alpha=0.6, label="facilitation")
plt.xlabel("drift after 6,000 steps (rad)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_drift():
    "without facilitation the median drift exceeds 0.3 rad"
    assert np.median(drift[False]) > 0.3
def test_stf():
    "facilitation cuts the median drift by more than half"
    assert np.median(drift[True]) < 0.5 * np.median(drift[False])

test_drift()
test_stf()
print("All checks passed")

### Exercise: Part B: which model generated the data?

Write nll_slots(p, data) and nll_resource(p, data): the summed negative log-likelihood of the errors over set sizes, using a normal density (wrapped onto the circle is not needed for these precisions) mixed with a uniform for slots. Fit each with scipy.optimize.minimize and compute BIC = 2 NLL + k ln n.

In [ ]:
from scipy.optimize import minimize
g = np.random.default_rng(2012)
wrap = lambda x: np.angle(np.exp(1j * x))
sd1 = np.deg2rad(15); sizes = [1, 2, 4, 6]
def gen_slots(N, n=400, K=3):
    st = g.random(n) < min(1, K / N)
    return np.where(st, sd1 * g.normal(size=n), g.uniform(-np.pi, np.pi, n))
def gen_resource(N, n=400):
    return wrap(sd1 * np.sqrt(N) * g.normal(size=n))
datasets = {"slot data": {N: gen_slots(N) for N in sizes}, "resource data": {N: gen_resource(N) for N in sizes}}
normal = lambda e, s: np.exp(-e ** 2 / (2 * s ** 2)) / np.sqrt(2 * np.pi * s ** 2)

def nll_slots(p, data):
    K, s = p
    ...

def nll_resource(p, data):
    s1 = p[0]
    ...

bic = {}
for name, data in datasets.items():
    n = sum(len(v) for v in data.values())
    fs = minimize(nll_slots, [2.5, 0.3], args=(data,), method="Nelder-Mead")
    fr = minimize(nll_resource, [0.3], args=(data,), method="L-BFGS-B", bounds=[(0.05, 2)])
    bic[name] = {"slots": 2 * fs.fun + 2 * np.log(n), "resource": 2 * fr.fun + 1 * np.log(n)}
    if name == "slot data":
        K_hat = fs.x[0]
    print(f"{name}: slots K = {fs.x[0]:.2f}, sd = {np.rad2deg(fs.x[1]):.1f} deg, BIC {bic[name]['slots']:.0f};  resource sd1 = {np.rad2deg(fr.x[0]):.1f} deg, BIC {bic[name]['resource']:.0f}")

In [ ]:
import numpy as np
def test_slot_data():
    "BIC prefers the slot model for slot data"
    assert bic["slot data"]["slots"] < bic["slot data"]["resource"]
def test_resource_data():
    "BIC prefers the resource model for resource data"
    assert bic["resource data"]["resource"] < bic["resource data"]["slots"]
def test_k():
    "the slot fit to slot data recovers K close to 3"
    assert abs(K_hat - 3) < 0.5

test_slot_data()
test_resource_data()
test_k()
print("All checks passed")